# 🧑‍💻 Face Detection & Recognition — End-to-End Pipeline

### Real-Time Face Recognition using **YOLOv8-Face** (Detection + Tracking) & **FaceNet / InceptionResnetV1** (Recognition)

**Author:** Om Choksi
**Notebook type:** End-to-end, cell-by-cell, Kaggle & Google Colab compatible
**Pipeline:** `Video/Camera → YOLOv8-Face Detector+Tracker → Face Crop → FaceNet Embedding → Cosine Similarity Match → Identity + Temporal Voting Buffer → Annotated Output Video`

---

**What this notebook does:**
1. Sets up the environment (auto-detects Kaggle vs Colab vs local)
2. Loads a YOLOv8 face detector (with keypoints) + a pretrained FaceNet (`vggface2`) recognizer
3. Builds a face embedding database from reference images
4. Runs a real-time-style recognition pipeline on a video (with an optional temporal voting buffer for stability)
5. **Evaluates** the system with proper metrics — ROC/AUC, EER, Accuracy/Precision/Recall/F1, Confusion Matrix, and similarity-score distributions — with plots
6. Summarizes results in a single results table

> This is a restructured, reproducible version of an original exploratory face-recognition prototype, reorganized into clean, documented, runnable sections.

## 📑 Table of Contents
1. [Environment Setup & Installation](#1)
2. [Imports](#2)
3. [Configuration & Path Setup (Kaggle / Colab compatible)](#3)
4. [Required Resources (weights & data)](#4)
5. [Device & Model Initialization](#5)
6. [Build Face Database (Embeddings)](#6)
7. [Face Embedding Utility Function](#7)
8. [Face Recognition Pipeline (Core Function)](#8)
9. [Run the Pipeline on Video](#9)
10. [Evaluation Metrics](#10)
11. [Visualizations](#11)
12. [Results Summary](#12)
13. [Conclusion & Future Work](#13)

<a id="1"></a>
## 1. Environment Setup & Installation

Works on **Kaggle**, **Google Colab**, or a **local/GPU server**. Run this cell once per session.

In [ ]:
%%capture
# Core dependencies for detection + recognition
!pip install ultralytics -q
!pip install facenet-pytorch -q
!pip install scikit-learn matplotlib seaborn -q


In [ ]:
# Quick GPU check (works on both Kaggle and Colab runtimes)
!nvidia-smi || echo "No GPU detected — the notebook will fall back to CPU (slower)."


<a id="2"></a>
## 2. Imports

In [ ]:
import os
import cv2
import math
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, Counter, defaultdict
from itertools import combinations

import torch
from torchvision import transforms
from PIL import Image
from facenet_pytorch import InceptionResnetV1
from ultralytics import YOLO

from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print("✅ All libraries imported successfully.")


<a id="3"></a>
## 3. Configuration & Path Setup (Kaggle / Colab compatible)

All tunable parameters and file paths live in **one place**. The notebook auto-detects whether it is
running on **Kaggle**, **Colab**, or **locally**, and sets sensible default directories for each — edit
`DATABASE_DIR`, `VIDEO_PATH` and `YOLO_WEIGHTS_PATH` to point at your own uploaded data/dataset.

In [ ]:
def detect_environment():
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV}")

if ENV == "kaggle":
    # On Kaggle, mount your dataset under /kaggle/input/<dataset-name>/...
    # and write outputs to /kaggle/working/
    BASE_DIR = "/kaggle/working"
    INPUT_DIR = "/kaggle/input"          # add your dataset here via "Add Data"
    DATABASE_DIR = f"{INPUT_DIR}/face-database/database"      # <-- update to your dataset folder
    VIDEO_PATH = f"{INPUT_DIR}/face-video/subway.mp4"          # <-- update to your dataset folder
    YOLO_WEIGHTS_PATH = f"{INPUT_DIR}/yolov8-face-weights/yolov8n-face-keypoints.pt"  # <-- update
elif ENV == "colab":
    from google.colab import drive
    BASE_DIR = "/content"
    DATABASE_DIR = "/content/database"
    VIDEO_PATH = "/content/subway.mp4"
    YOLO_WEIGHTS_PATH = "/content/yolov8n-face-keypoints.pt"
else:
    BASE_DIR = "."
    DATABASE_DIR = "./database"
    VIDEO_PATH = "./subway.mp4"
    YOLO_WEIGHTS_PATH = "./yolov8n-face-keypoints.pt"

OUTPUT_VIDEO_PATH = f"{BASE_DIR}/result.mp4"

# ---- Pipeline hyperparameters ----
DET_CONF_THRESHOLD = 0.6      # min YOLO detection confidence to accept a face box
SIM_DIFF_THRESHOLD = 0.7      # max (1 - cosine similarity) to accept an identity match
USE_TEMPORAL_BUFFER = True    # majority-vote over the last N crops per tracked face
BUFFER_SIZE = 5

print(f"BASE_DIR            = {BASE_DIR}")
print(f"DATABASE_DIR         = {DATABASE_DIR}")
print(f"VIDEO_PATH           = {VIDEO_PATH}")
print(f"YOLO_WEIGHTS_PATH    = {YOLO_WEIGHTS_PATH}")
print(f"OUTPUT_VIDEO_PATH    = {OUTPUT_VIDEO_PATH}")


<a id="4"></a>
## 4. Required Resources (Weights & Data)

You need three things on disk before running the pipeline cells below:

| Resource | Description | Where to get it |
|---|---|---|
| `YOLO_WEIGHTS_PATH` | A YOLOv8 face-detection model with keypoints (e.g. `yolov8n-face-keypoints.pt`) | Upload to Colab (`/content`) or attach as a Kaggle dataset |
| `DATABASE_DIR` | Folder of reference face images, one or more `.jpg` per identity, named `identity_xxx.jpg` (e.g. `om_01.jpg`) | Your own labeled photos |
| `VIDEO_PATH` | A test video to run recognition on | Your own clip, or a public sample video |

On **Colab**, uncomment and run the cell below to upload files interactively.
On **Kaggle**, use **"+ Add Data"** in the right-hand panel to attach a dataset/model, then update the paths in Section 3.

In [ ]:
if ENV == "colab":
    from google.colab import files
    # Uncomment the lines you need:
    # print("Upload YOLO face-detection weights (.pt):")
    # uploaded = files.upload()
    # print("Upload a zip of your face database, then unzip it into DATABASE_DIR:")
    # uploaded = files.upload()
    # print("Upload your test video:")
    # uploaded = files.upload()
    pass
else:
    print("Not running on Colab — attach your dataset/model via Kaggle 'Add Data', "
          "or place files locally at the paths configured in Section 3.")


<a id="5"></a>
## 5. Device & Model Initialization

- **Detector:** YOLOv8-face (with facial keypoints) — also used for lightweight multi-object **tracking** across frames
- **Recognizer:** `InceptionResnetV1` pretrained on **VGGFace2**, producing 512-d L2-normalized face embeddings

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

face_detector = YOLO(YOLO_WEIGHTS_PATH)
face_detector.to(device)

face_recognizer = InceptionResnetV1(pretrained="vggface2").eval().to(device)

print("✅ Face detector and recognizer loaded.")


<a id="6"></a>
## 6. Build Face Database (Embeddings)

Every reference image in `DATABASE_DIR` is embedded once and cached in memory as `Data_Base`.
Identity labels are parsed from the filename prefix before the first underscore
(e.g. `om_01.jpg` → identity `om`).

In [ ]:
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def build_face_database(database_dir):
    '''Embed every .jpg in `database_dir` and return (embeddings_tensor, filenames, identities).'''
    img_names_ls = sorted([f for f in os.listdir(database_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    embeddings = []

    for image_name in img_names_ls:
        image_path = os.path.join(database_dir, image_name)
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = face_recognizer(img_tensor)
            emb = emb / emb.norm(dim=1)
        embeddings.append(emb)

    data_base = torch.cat(embeddings, dim=0) if embeddings else torch.empty(0)
    identities = [name.split("_")[0] for name in img_names_ls]
    return data_base, img_names_ls, identities

Data_Base, img_names_ls, identities = build_face_database(DATABASE_DIR)
print(f"Database built: {Data_Base.shape[0]} images, {len(set(identities))} unique identities.")


<a id="7"></a>
## 7. Face Embedding Utility Function

Converts a raw BGR face crop (as returned by OpenCV/YOLO) into a normalized 512-d embedding.

In [ ]:
def face_crop_embedder(cropped_face: np.ndarray) -> torch.Tensor:
    '''Embed a single cropped face (BGR numpy array) into a unit-norm FaceNet embedding.'''
    cropped_face_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
    cropped_face_pil = Image.fromarray(cropped_face_rgb)
    cropped_face_tensor = transform(cropped_face_pil)

    with torch.no_grad():
        cropped_face_embedding = face_recognizer(cropped_face_tensor.unsqueeze(0).to(device))
        cropped_face_embedding = cropped_face_embedding / cropped_face_embedding.norm(dim=1)

    return cropped_face_embedding


<a id="8"></a>
## 8. Face Recognition Pipeline (Core Function)

One unified, parameterized function replaces the two original ad-hoc scripts
("without buffer" / "with buffer"). Set `use_buffer=False` to label every accepted frame
immediately, or `use_buffer=True` to smooth identities with a majority vote over the last
`buffer_size` recognitions per tracked face (recommended — much more stable on video).

In [ ]:
def run_face_recognition_pipeline(
    video_path,
    output_path,
    data_base,
    img_names_ls,
    identities,
    det_conf_threshold=DET_CONF_THRESHOLD,
    sim_diff_threshold=SIM_DIFF_THRESHOLD,
    use_buffer=USE_TEMPORAL_BUFFER,
    buffer_size=BUFFER_SIZE,
):
    '''
    Run YOLO detection+tracking -> FaceNet recognition on a video, write an annotated
    output video, and return per-detection stats for downstream evaluation.
    '''
    cap = cv2.VideoCapture(video_path)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    cam_fps = cap.get(cv2.CAP_PROP_FPS) or 15

    font = cv2.FONT_HERSHEY_SIMPLEX
    fontScale, thickness = 0.8, 2

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), cam_fps, (frame_width, frame_height))

    person_buffers = {}          # track_id -> deque of recent identity labels
    stats = {
        "det_confidences": [],   # every accepted detection's YOLO confidence
        "match_similarities": [],# best-match (1 - cosine) score for every accepted detection
        "matched": [],           # bool: was a match accepted under the threshold
        "frame_count": 0,
    }

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        stats["frame_count"] += 1

        results = face_detector.track(frame, persist=True, device=device)

        for r in results[0]:
            if r.boxes.conf.item() <= det_conf_threshold:
                continue

            stats["det_confidences"].append(r.boxes.conf.item())

            x_c, y_c, w_b, h_b = r.boxes.xywh.cpu().numpy()[0]
            x_min, x_max = int(x_c - w_b / 2), int(x_c + w_b / 2)
            y_min, y_max = int(y_c - h_b / 2), int(y_c + h_b / 2)
            cropped_face = frame[y_min:y_max, x_min:x_max]
            if cropped_face.size == 0:
                continue

            cropped_face_embedding = face_crop_embedder(cropped_face)
            similarities = 1 - (data_base * cropped_face_embedding).sum(dim=1)
            min_diff_index = torch.argmin(similarities)
            best_score = similarities[min_diff_index].item()

            stats["match_similarities"].append(best_score)
            is_match = best_score < sim_diff_threshold
            stats["matched"].append(is_match)

            display_label = None
            if use_buffer:
                face_id = r.boxes.id.item() if r.boxes.id is not None else -1
                if face_id not in person_buffers:
                    person_buffers[face_id] = deque(maxlen=buffer_size)
                if is_match:
                    person_buffers[face_id].append(identities[min_diff_index])
                if len(person_buffers[face_id]) == buffer_size:
                    display_label = Counter(person_buffers[face_id]).most_common(1)[0][0]
            else:
                if is_match:
                    display_label = img_names_ls[min_diff_index]

            if display_label:
                cv2.putText(frame, str(display_label), (int(x_c + w_b / 2), int(y_c)),
                            font, fontScale, (0, 0, 200), thickness, cv2.LINE_AA)
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    return stats


<a id="9"></a>
## 9. Run the Pipeline on Video

In [ ]:
pipeline_stats = run_face_recognition_pipeline(
    video_path=VIDEO_PATH,
    output_path=OUTPUT_VIDEO_PATH,
    data_base=Data_Base,
    img_names_ls=img_names_ls,
    identities=identities,
)

print(f"Processed {pipeline_stats['frame_count']} frames.")
print(f"Total accepted face detections: {len(pipeline_stats['det_confidences'])}")
print(f"Output video saved to: {OUTPUT_VIDEO_PATH}")


In [ ]:
# Preview one annotated frame inline (works on both Kaggle and Colab)
preview_cap = cv2.VideoCapture(OUTPUT_VIDEO_PATH)
preview_cap.set(cv2.CAP_PROP_POS_FRAMES, preview_cap.get(cv2.CAP_PROP_FRAME_COUNT) // 2)
ok, preview_frame = preview_cap.read()
preview_cap.release()

if ok:
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Sample annotated frame from output video")
    plt.show()
else:
    print("Could not read a preview frame — check that VIDEO_PATH/OUTPUT_VIDEO_PATH are valid.")


<a id="10"></a>
## 10. Evaluation Metrics

Since raw surveillance video rarely comes with frame-level identity ground truth, this notebook
evaluates the **recognition sub-system** the standard way biometric systems are benchmarked, directly
from the labeled face database built in Section 6:

- **Verification task** — for every pair of database embeddings, is "same identity" correctly
  separated from "different identity" by the similarity score? → **ROC curve, AUC, EER, Accuracy/Precision/Recall/F1 at the operating threshold**
- **Identification task** — leave-one-image-out: for each database image, treat it as a probe and
  find its nearest neighbor among the rest → **Confusion matrix, closed-set identification accuracy**
- **Detector behavior** — distribution of YOLO confidence scores and match scores observed on the actual test video (Section 9)

### 10.1 Build Genuine / Impostor Similarity Pairs

In [ ]:
def build_verification_pairs(data_base, identities):
    '''All unique embedding pairs -> (similarity, is_genuine) for ROC/EER analysis.'''
    n = data_base.shape[0]
    sims, labels = [], []
    for i, j in combinations(range(n), 2):
        sim = torch.dot(data_base[i], data_base[j]).item()
        sims.append(sim)
        labels.append(int(identities[i] == identities[j]))
    return np.array(sims), np.array(labels)

sims, gt_labels = build_verification_pairs(Data_Base, identities)
print(f"Built {len(sims)} verification pairs "
      f"({gt_labels.sum()} genuine, {len(gt_labels) - gt_labels.sum()} impostor).")

if gt_labels.sum() == 0 or gt_labels.sum() == len(gt_labels):
    print("⚠️ Need at least 2 identities AND at least one identity with 2+ images "
          "in DATABASE_DIR to compute genuine/impostor pairs.")


### 10.2 Verification Metrics — ROC, AUC, EER, Accuracy/Precision/Recall/F1

In [ ]:
# Similarity score -> higher = more likely genuine, so feed it directly to roc_curve
fpr, tpr, roc_thresholds = roc_curve(gt_labels, sims)
auc_score = roc_auc_score(gt_labels, sims)

# Equal Error Rate: point where FPR == FNR (1 - TPR)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = roc_thresholds[eer_idx]

# Metrics at the notebook's configured operating threshold
# (pipeline uses `1 - similarity < SIM_DIFF_THRESHOLD`  <=>  `similarity > 1 - SIM_DIFF_THRESHOLD`)
operating_sim_threshold = 1 - SIM_DIFF_THRESHOLD
preds_at_threshold = (sims > operating_sim_threshold).astype(int)

acc = accuracy_score(gt_labels, preds_at_threshold)
prec = precision_score(gt_labels, preds_at_threshold, zero_division=0)
rec = recall_score(gt_labels, preds_at_threshold, zero_division=0)
f1 = f1_score(gt_labels, preds_at_threshold, zero_division=0)

print(f"AUC                         : {auc_score:.4f}")
print(f"EER                         : {eer:.4f}  (at similarity threshold {eer_threshold:.4f})")
print(f"Operating similarity thresh.: {operating_sim_threshold:.4f}  (from SIM_DIFF_THRESHOLD={SIM_DIFF_THRESHOLD})")
print(f"Accuracy  @ operating thresh: {acc:.4f}")
print(f"Precision @ operating thresh: {prec:.4f}")
print(f"Recall    @ operating thresh: {rec:.4f}")
print(f"F1-score  @ operating thresh: {f1:.4f}")


### 10.3 Identification Metrics — Leave-One-Out Confusion Matrix

In [ ]:
def leave_one_out_identification(data_base, identities):
    '''For each embedding, find nearest neighbor among all others; return true/pred identity lists.'''
    n = data_base.shape[0]
    y_true, y_pred = [], []
    for i in range(n):
        sims_i = (data_base * data_base[i]).sum(dim=1)
        sims_i[i] = -1e9  # exclude self-match
        best_j = torch.argmax(sims_i).item()
        y_true.append(identities[i])
        y_pred.append(identities[best_j])
    return y_true, y_pred

y_true, y_pred = leave_one_out_identification(Data_Base, identities)
identification_accuracy = accuracy_score(y_true, y_pred)
print(f"Closed-set leave-one-out identification accuracy: {identification_accuracy:.4f}")

labels_sorted = sorted(set(identities))
cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)


<a id="11"></a>
## 11. Visualizations

All plots needed to sanity-check and showcase model performance in one place.

**11.1 Genuine vs. Impostor Similarity Score Distribution**

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.histplot(sims[gt_labels == 1], color="seagreen", label="Genuine pairs", kde=True, stat="density", alpha=0.5)
sns.histplot(sims[gt_labels == 0], color="indianred", label="Impostor pairs", kde=True, stat="density", alpha=0.5)
plt.axvline(operating_sim_threshold, color="black", linestyle="--", label="Operating threshold")
plt.xlabel("Cosine similarity")
plt.ylabel("Density")
plt.title("Genuine vs. Impostor Similarity Distribution")
plt.legend()
plt.tight_layout()
plt.show()


**11.2 ROC Curve**

In [ ]:
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color="royalblue", label=f"ROC (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.scatter([fpr[eer_idx]], [tpr[eer_idx]], color="red", zorder=5, label=f"EER = {eer:.3f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Face Verification")
plt.legend()
plt.tight_layout()
plt.show()


**11.3 Precision–Recall Curve**

In [ ]:
precisions, recalls, _ = precision_recall_curve(gt_labels, sims)

plt.figure(figsize=(5.5, 5))
plt.plot(recalls, precisions, color="darkorange")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve — Face Verification")
plt.tight_layout()
plt.show()


**11.4 Identification Confusion Matrix**

In [ ]:
fig, ax = plt.subplots(figsize=(max(5, 0.6 * len(labels_sorted) + 2), max(5, 0.6 * len(labels_sorted) + 2)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_sorted)
disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title("Leave-One-Out Identification Confusion Matrix")
plt.tight_layout()
plt.show()


**11.5 Detector Confidence & Match-Score Distribution (from the Section 9 video run)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.histplot(pipeline_stats["det_confidences"], bins=20, color="slateblue", ax=axes[0])
axes[0].set_title("YOLO Detection Confidence\n(accepted face boxes)")
axes[0].set_xlabel("Confidence")

sns.histplot(pipeline_stats["match_similarities"], bins=20, color="teal", ax=axes[1])
axes[1].axvline(SIM_DIFF_THRESHOLD, color="black", linestyle="--", label="SIM_DIFF_THRESHOLD")
axes[1].set_title("Best-Match Score Distribution\n(1 - cosine similarity)")
axes[1].set_xlabel("1 - cosine similarity")
axes[1].legend()

plt.tight_layout()
plt.show()


**11.6 Threshold Sweep — Accuracy vs. Similarity Threshold**

Shows how the verification accuracy would change if `SIM_DIFF_THRESHOLD` were tuned differently,
helping justify the chosen operating point.

In [ ]:
threshold_range = np.linspace(sims.min(), sims.max(), 50)
accs = [accuracy_score(gt_labels, (sims > t).astype(int)) for t in threshold_range]

plt.figure(figsize=(7, 4.5))
plt.plot(threshold_range, accs, color="crimson")
plt.axvline(operating_sim_threshold, color="black", linestyle="--", label="Current operating threshold")
plt.xlabel("Similarity threshold")
plt.ylabel("Verification accuracy")
plt.title("Accuracy vs. Similarity Threshold")
plt.legend()
plt.tight_layout()
plt.show()


<a id="12"></a>
## 12. Results Summary

In [ ]:
results_summary = pd.DataFrame({
    "Metric": [
        "Database size (images)",
        "Unique identities",
        "Verification AUC",
        "Verification EER",
        "Accuracy @ operating threshold",
        "Precision @ operating threshold",
        "Recall @ operating threshold",
        "F1-score @ operating threshold",
        "Closed-set identification accuracy",
        "Video frames processed",
        "Total accepted face detections",
    ],
    "Value": [
        Data_Base.shape[0],
        len(set(identities)),
        round(auc_score, 4),
        round(eer, 4),
        round(acc, 4),
        round(prec, 4),
        round(rec, 4),
        round(f1, 4),
        round(identification_accuracy, 4),
        pipeline_stats["frame_count"],
        len(pipeline_stats["det_confidences"]),
    ],
})
results_summary


In [ ]:
# Save results to disk for later reference / submission
results_path = f"{BASE_DIR}/evaluation_results.csv"
results_summary.to_csv(results_path, index=False)
print(f"Saved evaluation summary to: {results_path}")


<a id="13"></a>
## 13. Conclusion & Future Work

**Summary:** This notebook implements an end-to-end face detection and recognition pipeline combining
a YOLOv8 face detector (with tracking) and a FaceNet (VGGFace2) recognizer, evaluated with standard
verification (ROC/AUC/EER) and identification (confusion matrix) metrics on the reference face database,
plus detector/match-score diagnostics from an actual video run.

**Possible next steps:**
- Swap the leave-one-out database evaluation for a proper held-out labeled test set as more data becomes available
- Add face alignment (via detected keypoints) before embedding, to improve recognition robustness to pose
- Explore `MTCNN` or a `casia-webface`-pretrained recognizer as alternatives (see original prototype notes)
- Package the pipeline into a small CLI / API for reuse outside notebooks

---
**Notebook by Om Choksi**
🔗 GitHub: [OMCHOKSI108](https://github.com/OMCHOKSI108) · 🌐 [omchoksi.codes](https://omchoksi.codes) · ✍️ Medium blog